# Notebook de pruebas ampliado — WikipediaAnalytics

Este notebook hace dos tipos de comprobaciones sobre `WikipediaAnalytics.py`:

1. **Pruebas básicas** sobre los métodos `select_row_by_value`, `get_columns` y `aggregate_column`.
2. **Pruebas ampliadas de scraping** sobre los **5 HTML** de la carpeta `Ficheros`.

Está pensado para ejecutarse en Colab o en local. Busca los HTML tanto en el directorio actual como en `Ficheros/`.


In [1]:

import os
import math
import importlib
import pandas as pd
import numpy as np

import WikipediaAnalytics
importlib.reload(WikipediaAnalytics)
from WikipediaAnalytics import WikipediaAnalytics


In [2]:

def resolve_html_paths():
    candidates = [
        "serbia_es.html",
        "polonia_es.html",
        "espania_es.html",
        "grecia_es.html",
        "italia_es.html",
    ]

    resolved = []
    for name in candidates:
        if os.path.exists(name):
            resolved.append(name)
        elif os.path.exists(os.path.join("Ficheros", name)):
            resolved.append(os.path.join("Ficheros", name))
        else:
            raise FileNotFoundError(f"No encuentro el HTML: {name}")
    return resolved

ARCHIVOS_5 = resolve_html_paths()
ARCHIVOS_5


['serbia_es.html',
 'polonia_es.html',
 'espania_es.html',
 'grecia_es.html',
 'italia_es.html']

## 1. Pruebas básicas con un DataFrame controlado

In [3]:

def test_basic_methods():
    df_test = pd.DataFrame({
        "Country Name": ["Serbia", "Polonia", "España"],
        "Area (KM^2)": [88361.0, 312696.0, 505944.0],
        "Water (%)": [np.nan, 2.60, 1.04],
        "Population (hab.)": [6690887.0, 41026068.0, 49570725.0],
        "Density (hab./km^2)": [115.00, 123.20, 97.47],
        "GDP ($)": [4.7654e10, 8.42172e11, 1.73e12],
        "Last Event Date": pd.to_datetime([
            "2006-06-03", "1989-12-29", "1978-12-29"
        ]),
        "Latitude (º)": [44.818, 52.230, 40.417],
        "Longitude(º)": [20.457, 21.011, -3.703]
    })

    wa = WikipediaAnalytics(["dummy"])
    wa.df = df_test.copy()

    print("📌 Testing select_row_by_value...")
    row = wa.select_row_by_value("Country Name", "España")
    assert row is not None
    assert row.iloc[0]["Population (hab.)"] == 49570725.0
    row = wa.select_row_by_value("Country Name", "Francia")
    assert row is None
    try:
        wa.select_row_by_value("NoExiste", "España")
        raise AssertionError("No lanzó ValueError con columna inválida")
    except ValueError:
        pass
    print("  ✔ OK")

    print("📌 Testing get_columns...")
    one_col = wa.get_columns("Country Name")
    assert list(one_col.columns) == ["Country Name"]
    many_cols = wa.get_columns(["Country Name", "Population (hab.)"])
    assert list(many_cols.columns) == ["Country Name", "Population (hab.)"]
    try:
        wa.get_columns(["Country Name", "NoExiste"])
        raise AssertionError("No lanzó ValueError con columnas inválidas")
    except ValueError:
        pass
    print("  ✔ OK")

    print("📌 Testing aggregate_column...")
    assert wa.aggregate_column("Density (hab./km^2)", "max") == 123.2
    assert wa.aggregate_column("Density (hab./km^2)", "min") == 97.47
    expected_mean_lat = (44.818 + 52.230 + 40.417) / 3
    assert abs(wa.aggregate_column("Latitude (º)", "mean") - expected_mean_lat) < 1e-9
    try:
        wa.aggregate_column("Latitude (º)", "median")
        raise AssertionError("No lanzó ValueError con operación inválida")
    except ValueError:
        pass
    try:
        wa.aggregate_column("NoExiste", "mean")
        raise AssertionError("No lanzó ValueError con columna inexistente")
    except ValueError:
        pass
    print("  ✔ OK")

    print("\n🎉 Pruebas básicas superadas.")

test_basic_methods()


📌 Testing select_row_by_value...
  ✔ OK
📌 Testing get_columns...
  ✔ OK
📌 Testing aggregate_column...
  ✔ OK

🎉 Pruebas básicas superadas.


## 2. Scrap real con los 5 HTML

In [4]:

wa = WikipediaAnalytics(ARCHIVOS_5)
wa.scrap()

print("Forma del DataFrame:", wa.df.shape)
display(wa.df)


DEBUG: coord_text = '44°49′04″N 20°27′25″E'
DEBUG: repr = '44°49′04″N 20°27′25″E'
DEBUG: coord_text = '52°13′48″N 21°00′40″E'
DEBUG: repr = '52°13′48″N 21°00′40″E'
DEBUG: coord_text = '40°25′01″N 3°42′12″O'
DEBUG: repr = '40°25′01″N 3°42′12″O'
DEBUG: coord_text = '37°59′03″N 23°43′41″E'
DEBUG: repr = '37°59′03″N 23°43′41″E'
DEBUG: coord_text = '41°53′35″N 12°28′58″E'
DEBUG: repr = '41°53′35″N 12°28′58″E'
Forma del DataFrame: (5, 9)


,Country Name,Area (KM^2),Water (%),Population (hab.),Density (hab./km^2),GDP ($),Last Event Date,Latitude (º),Longitude (º)
0,Serbia,88361.0,NaN,6690887.0,115.00,4.765400e+10,2006-06-03,44.817778,20.456944
1,Polonia,312696.0,26.00,41026068.0,1232.00,8.421720e+11,1989-12-29,52.230000,21.011111
2,España,505944.0,1.04,49570725.0,97.47,1.730000e+12,1978-12-29,40.416944,-3.703333
3,Grecia,131957.0,0.84,10432481.0,79.06,2.423850e+11,1974-07-24,37.984167,23.728056
4,Italia,302073.0,2.40,58915561.0,195.50,2.058000e+09,1946-06-02,41.893056,12.482778


In [5]:

EXPECTED = {
    "Serbia": {
        "Area (KM^2)": 88361.0,
        "Water (%)": np.nan,
        "Population (hab.)": 6690887.0,
        "Density (hab./km^2)": 115.0,
        "Last Event Date": pd.Timestamp("2006-06-03"),
        "Latitude (º)": 44.818,
        "Longitude(º)": 20.457,
    },
    "Polonia": {
        "Area (KM^2)": 312696.0,
        "Water (%)": 2.6,
        "Population (hab.)": 41026068.0,
        "Density (hab./km^2)": 123.2,
        "Last Event Date": pd.Timestamp("1989-12-29"),
        "Latitude (º)": 52.230,
        "Longitude(º)": 21.011,
    },
    "España": {
        "Area (KM^2)": 505944.0,
        "Water (%)": 1.04,
        "Population (hab.)": 49570725.0,
        "Density (hab./km^2)": 97.47,
        "Last Event Date": pd.Timestamp("1978-12-29"),
        "Latitude (º)": 40.417,
        "Longitude(º)": -3.703,
    },
    "Grecia": {
        "Area (KM^2)": 131957.0,
        "Water (%)": 0.84,
        "Population (hab.)": 10432481.0,
        "Density (hab./km^2)": 79.06,
        "Last Event Date": pd.Timestamp("1974-07-24"),
        "Latitude (º)": 37.984,
        "Longitude(º)": 23.728,
    },
    "Italia": {
        "Area (KM^2)": 302073.0,
        "Water (%)": 2.4,
        "Population (hab.)": 58915561.0,
        "Density (hab./km^2)": 195.5,
        "Last Event Date": pd.Timestamp("1946-06-02"),
        "Latitude (º)": 41.893,
        "Longitude(º)": 12.483,
    },
}
EXPECTED


{'Serbia': {'Area (KM^2)': 88361.0,
  'Water (%)': nan,
  'Population (hab.)': 6690887.0,
  'Density (hab./km^2)': 115.0,
  'Last Event Date': Timestamp('2006-06-03 00:00:00'),
  'Latitude (º)': 44.818,
  'Longitude(º)': 20.457},
 'Polonia': {'Area (KM^2)': 312696.0,
  'Water (%)': 2.6,
  'Population (hab.)': 41026068.0,
  'Density (hab./km^2)': 123.2,
  'Last Event Date': Timestamp('1989-12-29 00:00:00'),
  'Latitude (º)': 52.23,
  'Longitude(º)': 21.011},
 'España': {'Area (KM^2)': 505944.0,
  'Water (%)': 1.04,
  'Population (hab.)': 49570725.0,
  'Density (hab./km^2)': 97.47,
  'Last Event Date': Timestamp('1978-12-29 00:00:00'),
  'Latitude (º)': 40.417,
  'Longitude(º)': -3.703},
 'Grecia': {'Area (KM^2)': 131957.0,
  'Water (%)': 0.84,
  'Population (hab.)': 10432481.0,
  'Density (hab./km^2)': 79.06,
  'Last Event Date': Timestamp('1974-07-24 00:00:00'),
  'Latitude (º)': 37.984,
  'Longitude(º)': 23.728},
 'Italia': {'Area (KM^2)': 302073.0,
  'Water (%)': 2.4,
  'Population (

## 3. Pruebas de estructura y tipos

In [6]:

def test_structure_and_types(wa):
    df = wa.df.copy()

    expected_cols = [
        "Country Name",
        "Area (KM^2)",
        "Water (%)",
        "Population (hab.)",
        "Density (hab./km^2)",
        "GDP ($)",
        "Last Event Date",
        "Latitude (º)",
        "Longitude(º)",
    ]
    assert list(df.columns) == expected_cols, f"Columnas incorrectas: {list(df.columns)}"
    assert len(df) == 5, f"Se esperaban 5 filas y hay {len(df)}"

    assert pd.api.types.is_string_dtype(df["Country Name"]) or df["Country Name"].dtype == object
    for col in [
        "Area (KM^2)", "Water (%)", "Population (hab.)",
        "Density (hab./km^2)", "GDP ($)", "Latitude (º)", "Longitude(º)"
    ]:
        assert pd.api.types.is_numeric_dtype(df[col]), f"La columna {col} no es numérica"

    assert pd.api.types.is_datetime64_any_dtype(df["Last Event Date"]), "Last Event Date no es datetime"

    assert set(df["Country Name"]) == set(EXPECTED.keys()), "Los países no coinciden con los esperados"

    print("✅ Estructura y tipos correctos.")

test_structure_and_types(wa)


AssertionError: Columnas incorrectas: ['Country Name', 'Area (KM^2)', 'Water (%)', 'Population (hab.)', 'Density (hab./km^2)', 'GDP ($)', 'Last Event Date', 'Latitude (º)', 'Longitude (º)']

## 4. Pruebas exactas por país

In [7]:

def assert_close(actual, expected, tol=1e-3):
    if pd.isna(expected):
        assert pd.isna(actual), f"Se esperaba NaN y se obtuvo {actual}"
    else:
        assert abs(float(actual) - float(expected)) <= tol, f"Esperado {expected}, obtenido {actual}"

def test_exact_values(wa):
    df = wa.df.set_index("Country Name")

    for country, values in EXPECTED.items():
        assert country in df.index, f"No aparece {country} en el DataFrame"
        row = df.loc[country]

        for col, expected in values.items():
            actual = row[col]
            if isinstance(expected, pd.Timestamp):
                assert pd.Timestamp(actual) == expected, f"{country} - {col}: esperado {expected}, obtenido {actual}"
            else:
                assert_close(actual, expected)

    print("✅ Valores clave correctos para los 5 países.")

test_exact_values(wa)


KeyError: 'Longitude(º)'

## 5. Pruebas más complicadas

In [9]:

def test_advanced_checks(wa):
    df = wa.df.copy()

    # Orden esperado por población: España > Italia > Polonia > Grecia > Serbia
    pop_order = list(df.sort_values("Population (hab.)", ascending=False)["Country Name"])
    assert pop_order == ["Italia", "España", "Polonia", "Grecia", "Serbia"], pop_order

    # País con mayor densidad
    max_density_country = df.loc[df["Density (hab./km^2)"].idxmax(), "Country Name"]
    assert max_density_country == "Italia", max_density_country

    # País con menor densidad
    min_density_country = df.loc[df["Density (hab./km^2)"].idxmin(), "Country Name"]
    assert min_density_country == "Grecia", min_density_country

    # Único país con longitud negativa: España
    negatives = set(df.loc[df["Longitude(º)"] < 0, "Country Name"])
    assert negatives == {"España"}, negatives

    # Único país con agua NaN: Serbia
    nan_water = set(df.loc[df["Water (%)"].isna(), "Country Name"])
    assert nan_water == {"Serbia"}, nan_water

    # El evento histórico más reciente debe ser Serbia (2006)
    latest_country = df.loc[df["Last Event Date"].idxmax(), "Country Name"]
    assert latest_country == "Serbia", latest_country

    # Las medias de coordenadas deben ser coherentes
    expected_lat_mean = np.mean([44.818, 52.230, 40.417, 37.984, 41.893])
    expected_lon_mean = np.mean([20.457, 21.011, -3.703, 23.728, 12.483])
    assert abs(df["Latitude (º)"].mean() - expected_lat_mean) < 1e-6
    assert abs(df["Longitude(º)"].mean() - expected_lon_mean) < 1e-6

    # Prueba de API pública de la clase sobre el scrap real
    row = wa.select_row_by_value("Country Name", "Grecia")
    assert row is not None
    assert abs(row.iloc[0]["Density (hab./km^2)"] - 79.06) < 1e-6

    subset = wa.get_columns(["Country Name", "Latitude (º)", "Longitude(º)"])
    assert subset.shape == (5, 3)

    assert abs(wa.aggregate_column("Density (hab./km^2)", "max") - 195.5) < 1e-6
    assert abs(wa.aggregate_column("Density (hab./km^2)", "min") - 79.06) < 1e-6

    print("✅ Pruebas avanzadas superadas.")

test_advanced_checks(wa)


AssertionError: Polonia

## 6. Revisión rápida de posibles señales de fragilidad

In [10]:

def quick_sanity_report(wa):
    df = wa.df.copy()
    report = pd.DataFrame({
        "nulls": df.isna().sum(),
        "dtype": df.dtypes.astype(str),
    })
    display(report)

    print("\nResumen:")
    print("- Filas:", len(df))
    print("- Columnas:", len(df.columns))
    print("- Países:", ", ".join(df["Country Name"].tolist()))

quick_sanity_report(wa)


,nulls,dtype
Country Name,0,str
Area (KM^2),0,float64
Water (%),1,float64
Population (hab.),0,float64
Density (hab./km^2),0,float64
GDP ($),0,float64
Last Event Date,0,datetime64[us]
Latitude (º),0,float64
Longitude (º),0,float64



Resumen:
- Filas: 5
- Columnas: 9
- Países: Serbia, Polonia, España, Grecia, Italia


## 7. Nota sobre el PIB

He dejado el notebook centrado sobre todo en:
- estructura
- tipos
- coordenadas
- fechas
- superficie
- agua
- población
- densidad

porque en algunos HTML de Wikipedia la redacción del PIB mezcla formatos como `millones`, `mill. USD` o `billones`.
Si quieres, se puede añadir una batería específica de tests para `GDP ($)` una vez fijes exactamente la convención que vas a usar para normalizar esas unidades.


In [ ]:
from WikipediaAnalytics import WikipediaAnalytics

ARCHIVOS_5 = [
    "serbia_es.html",
    "polonia_es.html",
    "espania_es.html",
    "grecia_es.html",
    "italia_es.html"
]

In [ ]:
wa = WikipediaAnalytics(list(reversed(ARCHIVOS_5)))
wa.scrap()
print(wa.df[["Country Name", "Population (hab.)"]])

  Country Name  Population (hab.)
0       Italia         58915561.0
1       Grecia         10432481.0
2       España         49570725.0
3      Polonia         41026068.0
4       Serbia          6690887.0


In [ ]:
wa = WikipediaAnalytics(ARCHIVOS_5)
wa.scrap()

assert set(wa.df["Country Name"]) == {"Serbia", "Polonia", "España", "Grecia", "Italia"}

In [ ]:
assert isinstance(wa.aggregate_column("Population (hab.)", "max"), float)
assert isinstance(wa.aggregate_column("Density (hab./km^2)", "mean"), float)
assert isinstance(wa.aggregate_column("Latitude (º)", "min"), float)

In [ ]:
try:
    wa.get_columns("NoExiste")
    print("Fallo: no lanzó error")
except ValueError:
    print("OK")

try:
    wa.aggregate_column("Population (hab.)", "sum")
    print("Fallo: no lanzó error")
except ValueError:
    print("OK")

OK
OK


In [ ]:
print(wa.df[["Country Name", "Last Event Date"]])
assert wa.df["Last Event Date"].isna().sum() == 0

  Country Name Last Event Date
0       Serbia      2006-06-03
1      Polonia      1989-12-29
2       España      1978-12-29
3       Grecia      1974-07-24
4       Italia      1946-06-02


In [ ]:
esp = wa.select_row_by_value("Country Name", "España").iloc[0]
assert esp["Longitude(º)"] < 0

In [ ]:
wa = WikipediaAnalytics(["grecia_es.html", "italia_es.html"])
wa.scrap()
assert set(wa.df["Country Name"]) == {"Grecia", "Italia"}
assert wa.df.shape[0] == 2

In [ ]:
wa = WikipediaAnalytics(ARCHIVOS_5)
wa.scrap()

df_sorted = wa.df.sort_values("Country Name").reset_index(drop=True)
assert list(df_sorted["Country Name"]) == ["España", "Grecia", "Italia", "Polonia", "Serbia"]

In [ ]:
import random

archivos_random = ARCHIVOS_5.copy()
random.shuffle(archivos_random)

wa = WikipediaAnalytics(archivos_random)
wa.scrap()

assert set(wa.df["Country Name"]) == {"Serbia", "Polonia", "España", "Grecia", "Italia"}
print("OK orden aleatorio")

OK orden aleatorio


In [ ]:
for c in wa.df["Country Name"]:
    assert "\xa0" not in c
    assert "\u200b" not in c
    assert "\ufeff" not in c

print("OK caracteres invisibles")

OK caracteres invisibles


In [ ]:
wa = WikipediaAnalytics(ARCHIVOS_5)

wa.scrap()
df1 = wa.df.copy()

wa.scrap()
df2 = wa.df.copy()

assert df1.equals(df2)

print("OK scrap() idempotente")

OK scrap() idempotente


In [ ]:
wa = WikipediaAnalytics(["espania_es.html"])
wa.scrap()

assert wa.df.shape == (1, 9)
assert wa.df.iloc[0]["Country Name"] == "España"
assert wa.df.iloc[0]["Longitude(º)"] < 0
print("OK un solo archivo")

OK un solo archivo


In [ ]:
wa = WikipediaAnalytics(["espania_es.html"])
wa.scrap()

assert wa.df.shape == (1, 9)
assert wa.df.iloc[0]["Country Name"] == "España"
assert wa.df.iloc[0]["Longitude(º)"] < 0
print("OK un solo archivo")

OK un solo archivo


In [ ]:
wa = WikipediaAnalytics(["espania_es.html", "espania_es.html"])
wa.scrap()

assert wa.df.shape[0] == 2
assert list(wa.df["Country Name"]) == ["España", "España"]
print("OK archivos duplicados")

OK archivos duplicados


In [ ]:
wa = WikipediaAnalytics(ARCHIVOS_5)
wa.scrap()

row = wa.select_row_by_value("Population (hab.)", 49570725.0)
assert row is not None
assert row.iloc[0]["Country Name"] == "España"
print("OK select_row_by_value numérico")

OK select_row_by_value numérico


In [ ]:
cols = wa.get_columns(["Country Name"])
assert list(cols.columns) == ["Country Name"]
print("OK get_columns lista de una columna")

OK get_columns lista de una columna


In [ ]:
try:
    wa.aggregate_column("Last Event Date", "mean")
    print("Ojo: acepta datetime")
except Exception:
    print("OK datetime no agregable")

OK datetime no agregable


In [ ]:
assert wa.df["Latitude (º)"].between(-90, 90).all()
assert wa.df["Longitude(º)"].between(-180, 180).all()
print("OK rangos de coordenadas")

OK rangos de coordenadas


In [ ]:
assert (wa.df["Population (hab.)"] > 0).all()
assert (wa.df["Density (hab./km^2)"] > 0).all()
assert (wa.df["Area (KM^2)"] > 0).all()
print("OK magnitudes positivas")

OK magnitudes positivas
